In [ ]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd()
data_dir = repo_root / "concat" if (repo_root / "concat").is_dir() else repo_root

accepted_path = data_dir / "accepted_2007_to_2018Q4.csv.gz"
rejected_path = data_dir / "rejected_2007_to_2018Q4.csv.gz"

if not accepted_path.exists():
    raise FileNotFoundError(f"Missing file: {accepted_path}")
if not rejected_path.exists():
    raise FileNotFoundError(f"Missing file: {rejected_path}")

accepted_df = pd.read_csv(accepted_path, low_memory=False)
rejected_df = pd.read_csv(rejected_path, low_memory=False)

desired_rows = 250_000
if len(accepted_df) < desired_rows:
    raise ValueError(f"accepted has only {len(accepted_df):,} rows")
if len(rejected_df) < desired_rows:
    raise ValueError(f"rejected has only {len(rejected_df):,} rows")
# Randomly select rows to avoid bias
accepted_sample = accepted_df.sample(n=desired_rows, random_state=42)
rejected_sample = rejected_df.sample(n=desired_rows, random_state=42)

combined_df = pd.concat([accepted_sample, rejected_sample], ignore_index=True, sort=False)

keep_cols = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_title",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "issue_d",
    "loan_status",
    "purpose",
    "title",
    "zip_code",
    "addr_state",
    "dti",
    "earliest_cr_line",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "initial_list_status",
    "application_type",
    "mort_acc",
    "pub_rec_bankruptcies",
]
missing_cols = [c for c in keep_cols if c not in combined_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in combined data: {missing_cols}")

combined_df = combined_df[keep_cols]

print("accepted:", accepted_sample.shape)
print("rejected:", rejected_sample.shape)
print("combined (kept cols):", combined_df.shape)

output_path = data_dir / "concat_accepted_rejected.csv.gz"
combined_df.to_csv(output_path, index=False, compression="gzip")
print("wrote:", output_path)

accepted: (250000, 152)
rejected: (250000, 10)
combined: (500000, 161)
wrote: e:\Credit Risk Data Science\lendingClubLoanData\concat\combined_accepted_rejected.csv.gz
